# New formatting -- from Matthew Kerr

The files, in `@kerrm/toby_files`

* `toby_v5.{npz,pickle}` -- this is the same format as before, with my current  version of the diffuse model.  NB that there is no longer a point sources template since the sources are modeled individually.
* `toby_ptsrcs_v5.npz` -- new


This latter file has the following entries:


* names -- the FL16Y names of the catalog sources.  They are sub-ordered by spectral model, so all of the PowerLaw sources appear first, then LogParabola, then PLEC4
* spectra -- this is a (nband=43,nsrc=7141) array giving the total counts in each band predicted for the FL16Y model values
* entries_per_band -- (nband=43) array giving the total number of ps model counts for each band
* pscounts -- (sum(entries_per_band)) array with the counts for each pixel for each point sources.
* healpixidx -- the data HEALPixel index for the entries of pscounts
* nameidx -- index into the names array identifying the source for each of the entries in the list


Within each band, the point source arrays are sorted by nameidx.

In [46]:
%reset -f
%run setup_notebook
%run pylib/tools dark date
from like3 import kerr_pixel_table as kpt; reload(kpt)
KerrPixelTable = kpt.KerrPixelTable
KerrPtsrcInfo = kpt.KerrPtsrcInfo

<h5 style="text-align:right; margin-right:15px"> 2026-05-15 12:25</h5>

### Load basic pixel table, maybe convert to FITS

In [47]:
kpt = KerrPixelTable('files/kerr/toby_v5')
# kpt.to_fits('files/kerr/toby_v5.fits')

Loaded columns ['diffuse', 'extendedsources', 'sunmoon', 'counts', 'indices'] from files/kerr/toby_v5.npz


### Read, examine the new file

In [59]:
import importlib
import numpy as np
import like3.kerr_pixel_table as kpt_mod

# Ensure the notebook picks up the newly added KerrPtsrcInfo methods.
importlib.reload(kpt_mod)
KerrPtsrcInfo = kpt_mod.KerrPtsrcInfo

self = ptsrc_info = KerrPtsrcInfo('files/kerr/toby_v5')

# Test the source-band collector on the first band with entries.
band_index = 0
band_row = ptsrc_info.meta.iloc[band_index]
sl = band_row['slice']

if band_row['entries'] == 0:
    raise RuntimeError('Band 0 has no entries to test')

# Pick a few sources present in this band.
source_indices = [int(v) for v in np.unique(ptsrc_info.nameidx[sl])[:3]]
if len(source_indices) < 2:
    raise RuntimeError('Need at least two sources in this band to test multi-source PtBand')

ptband = ptsrc_info.create_ptband(source_indices, band_index)

expected_cols = ['pixel_id'] + [f'counts_{s}' for s in source_indices]
print(ptband)
print('columns:', list(ptband.table.columns))
print('rows:', len(ptband.table))
print(ptband.table.head())

assert list(ptband.table.columns) == expected_cols
assert len(ptband.table) > 0
assert (ptband.table[expected_cols[1:]] >= 0).all().all()

# Backward compatibility: single-source call still returns a 'counts' column.
ptband_single = ptsrc_info.create_ptband(source_indices[0], band_index)
assert list(ptband_single.table.columns) == ['pixel_id', 'counts']

Loaded columns ['names', 'spectra', 'entries_per_band', 'nameidx', 'healpixidx', 'pscounts'] from files/kerr/toby_ptsrcs_v5.npz
Total number of pixels: 18,265,372
Loaded meta data describing 43 bands from files/kerr/toby_v5.pickle
PtBand(source=[0, 1, 2], band=0, pixels=450)
columns: ['pixel_id', 'counts_0', 'counts_1', 'counts_2']
rows: 450
   pixel_id  counts_0  counts_1  counts_2
0     38942       0.0  0.000686       0.0
1     38943       0.0  0.002291       0.0
2     38957       0.0  0.000262       0.0
3     38959       0.0  0.001813       0.0
4     38961       0.0  0.000945       0.0


<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   event_type  43 non-null     category
 1   emin        43 non-null     int64   
 2   emax        43 non-null     int64   
 3   nside       43 non-null     int64   
 4   entries     43 non-null     int64   
 5   energy      43 non-null     int64   
 6   slice       43 non-null     object  
dtypes: category(1), int64(5), object(1)
memory usage: 2.2+ KB


In [54]:
idx = list(self.names).index('FL16Y J0633.9+1746')

In [56]:
sum(self.nameidx==idx)

np.int64(3347)